*Cellule 1 — Vérifier l'environnement*

In [ ]:
import openai, pandas as pd

print("openai version:", openai.__version__)
print("pandas version:", pd.__version__)

*Cellule 2 — Connexion au serveur*

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

models = client.models.list()
print("Modèles disponibles :")
for m in models.data:
    print(" -", m.id)

*Cellule 3 — Un tout premier échange*

In [ ]:
response = client.chat.completions.create(
    model="mistralai/mistral-7b-instruct-v0.3",
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
print("Réponse :", response.choices[0].message.content)

*Cellule 4 — Un prompt un peu plus long (mi-chemin)*

In [ ]:
response = client.chat.completions.create(
    model="mistralai/mistral-7b-instruct-v0.3",
    messages=[
        {"role": "system", "content": "You are a supply chain analyst. Be concise and factual."},
        {"role": "user", "content": "A supplier has 15 delivery delays this month and 3 quality issues. Summarize in one sentence."},
    ],
    temperature=0.2,
    max_tokens=100,
)
print("Réponse :", response.choices[0].message.content)

*Cellule 5 — Charger tes données*

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../scripts"))

df = pd.read_csv("../data/cleaned_comments.csv")
print("Lignes :", len(df))

*Cellule 6 — Calculer les stats*

In [ ]:
from stats_engine import compute_stats

stats = compute_stats(df)
print("total_comments :", stats["total_comments"])
print("Clés :", list(stats.keys()))

*Cellule 7 — Construire le prompt (sans appeler le serveur)*

In [ ]:
from llm_summary import build_prompt

sample_comments = df["Comment_EN"].value_counts().head(5).index.tolist()
messages = build_prompt(stats, sample_comments)

print("Nb de messages :", len(messages))
print("Taille totale (caractères) :", sum(len(m["content"]) for m in messages))
print("\n--- PROMPT SYSTÈME ---\n", messages[0]["content"])
print("\n--- PROMPT UTILISATEUR (800 premiers caractères) ---\n", messages[1]["content"][:800])

*Cellule 8 — L'appel final (le vrai test)*

In [ ]:
from llm_summary import generate_summary

summary = generate_summary(stats, sample_comments)
print("\n=== AI SUMMARY ===")
print(summary)